In [1]:
"""
Retrieve-Then-Compress (RTC) RAG Pipeline -- Production-Grade
==============================================================
Architecture   : FIVE configurations covering the complete Retrieve-Then-Compress
                 design space as established by the 2023-2025 research literature:
                 (1) Baseline RTC                 -- raw retrieval, no compression
                 (2) RECOMP Extractive            -- sentence-level selection across
                                                    all retrieved docs jointly
                 (3) RECOMP Abstractive           -- multi-doc fusion summarization
                                                    into a single compressed context
                 (4) LLMLingua Token-Budget RTC   -- perplexity-guided token pruning
                                                    with hard token budget enforcement
                 (5) MapReduce + Evidentiality    -- map: per-doc extractive summaries,
                     Iterative RTC [BEST]            reduce: cross-doc synthesis,
                                                    evidentiality check loop, iterate
                                                    until evidence judged sufficient

Vector Store   : FAISS  (IndexFlatIP, BGE-large-en-v1.5, 1024-dim)
BM25           : BM25Plus (rank-bm25)
Embeddings     : BAAI/bge-large-en-v1.5  (1024-dim, query instruction prefix)
LLM            : Azure OpenAI  (ChatOllama -- GPT-4o)
Framework      : LangChain 0.2+  (ContextualCompressionRetriever,
                                  LLMLinguaCompressor, EnsembleRetriever)

Reference PDFs (open-access arXiv):
    1. "Attention Is All You Need" -- Vaswani et al., 2017
       https://arxiv.org/pdf/1706.03762.pdf
    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018
       https://arxiv.org/pdf/1810.04805.pdf
    3. "RAG for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020
       https://arxiv.org/pdf/2005.11401.pdf

=============================================================================
What Retrieve-Then-Compress Is and Why It Exists
=============================================================================
Retrieve-Then-Compress (RTC) is a RAG architecture paradigm in which retrieved
documents are compressed into shorter, denser representations BEFORE they are
injected into the LLM's context window for answer generation.

Standard RAG injects raw retrieved chunks verbatim into the LLM context.
At K=10 chunks of 450 characters each, the context contains approximately
4,500 characters (~1,125 tokens) of retrieved content. Research consistently
shows that only 10-20% of this content is directly relevant to the query.

RTC addresses three compounding problems that arise from uncompressed retrieval:

    PROBLEM 1 -- LOST-IN-THE-MIDDLE DEGRADATION:
        Long contexts suffer from a systematic attention failure where the LLM
        assigns disproportionately low attention weights to content in the MIDDLE
        of the context. A retrieved document at position 5-8 of 10 receives less
        attention than documents at position 1-2 or 9-10. Compression reduces the
        total context length, moving all relevant content closer to the boundaries
        where attention is strongest. LongLLMLingua mitigates the 'lost in the
        middle' issue in LLMs, enhancing long-context information processing,
        improving RAG performance by up to 21.4% using only 1/4 of the tokens.

    PROBLEM 2 -- CONTEXT COST INFLATION:
        At GPT-4o pricing ($2.50/M input tokens), 10,000 queries per day at 1,125
        tokens of retrieved context per query = 11.25M context tokens/day = $28.13/day
        from retrieval context alone. An 80% compression ratio reduces this to ~$5.63/day
        -- a $22.50/day saving ($8,212/year) from a single product feature.
        Research shows 70-90% token reduction while maintaining answer quality.

    PROBLEM 3 -- NOISE AMPLIFICATION AT GENERATION:
        Irrelevant context tokens do not merely waste the budget -- they actively
        degrade answer quality by providing the LLM with plausible-but-wrong
        information to draw on. A document about "attention mechanisms in general"
        alongside a query about "the specific BLEU score" gives the LLM multiple
        equally-plausible numbers to hallucinate from. Compression removes the
        irrelevant plausible-but-wrong content, reducing hallucination pathways.

=============================================================================
The RECOMP Framework (Xu et al., ICLR 2024)
=============================================================================
RECOMP introduces an intermediary step for Retrieval-augmented Language Models.
RECOMP compresses retrieved documents into concise textual summaries before
integrating them during inference, reducing computational costs and alleviating
the burden on LMs to process lengthy documents. The aim is to produce summaries
that balance brevity and fidelity to the original evidence documents, guiding the
RALM to produce targeted outputs when the summary is used as a prefix to the input.

RECOMP trains two types of compressors:

    EXTRACTIVE COMPRESSOR:
        Filters out irrelevant sentences, retaining only the most pertinent ones
        from the retrieved document set. Contrastive learning is employed to train
        the extractive compressor enabling it to select key sentences effectively.
        In the zero-shot LLM implementation (this pipeline), we prompt GPT-4o to
        perform the same role: given the query and all retrieved sentences jointly,
        identify which sentences should be retained in the compressed context.
        The extractive compressor returns verbatim source sentences -- no paraphrase,
        no synthesis. This guarantees source faithfulness.

    ABSTRACTIVE COMPRESSOR:
        Generates a summary by fusing information from multiple retrieved documents.
        The abstractive compressor is distilled from a large language model (GPT-3/4),
        achieving strong summarization performance. In the zero-shot implementation,
        GPT-4o is prompted to fuse all retrieved documents into a single compressed
        paragraph that contains all information needed to answer the query.
        Unlike extractive, abstractive can produce content that is NOT verbatim from
        any single source -- it synthesizes across documents. This enables higher
        information density but introduces a (small) hallucination risk.

SELECTIVE AUGMENTATION:
    Both RECOMP compressors implement selective augmentation: when retrieved documents
    are irrelevant to the input or offer no additional information, the compressor
    outputs an empty string instead of a compressed summary. This prevents the
    pipeline from injecting irrelevant context that would actively degrade the LLM.
    In this pipeline, the empty-output condition is detected and triggers a fallback
    to the raw top-2 documents to ensure the LLM always has some context.

=============================================================================
LLMLingua: Token-Budget-Aware Prompt Compression (EMNLP 2023)
=============================================================================
LLMLingua uses a compact, well-trained language model (e.g., GPT2-small, LLaMA-7B)
to identify and remove non-essential tokens in prompts. This approach enables
efficient inference with large language models (LLMs), achieving up to 20x
compression with minimal performance loss.

Key innovation -- PERPLEXITY-GUIDED TOKEN IMPORTANCE:
    LLMLingua leverages small models' perplexity to measure the redundancy within
    a prompt. Tokens that the small model can predict with LOW perplexity (high
    probability) are REDUNDANT -- they carry information already established by
    the surrounding context. Tokens with HIGH perplexity are IMPORTANT -- they
    carry information the model could not have predicted from context alone.
    The budget controller enforces a target compression ratio by setting a perplexity
    threshold: tokens above the threshold (harder to predict) are retained, tokens
    below are dropped.

    This differs fundamentally from LLM-based extraction (RECOMP/LLMChainExtractor):
        LLMLingua:  operates at the TOKEN level -- individual tokens are dropped.
                    Uses a small local model (~GPT2-small). Zero LLM API calls.
                    Output may be grammatically incorrect but semantically dense.
        RECOMP:     operates at the SENTENCE level -- whole sentences are kept or dropped.
                    Uses the answer LLM or a fine-tuned model.
                    Output is grammatically correct but at sentence granularity.

    pip install llmlingua to use LLMLinguaCompressor via LangChain.

=============================================================================
MapReduce for Multi-Document RAG Compression
=============================================================================
Recent innovation in context management includes MapReduce-inspired approaches
where the LLM first processes individual chunks (MAP), then synthesizes these
intermediate outputs into a final response (REDUCE). This technique has shown
promising results for complex queries requiring information from many documents.

MAP phase: For each retrieved document independently:
    - Summarize or extract the relevant content for the query.
    - Produce a compressed intermediate representation.
    - Each MAP call is independent and parallelizable.

REDUCE phase: Take all MAP outputs together:
    - Synthesize the per-document summaries into a single coherent context.
    - The REDUCE LLM sees the compressed intermediates, not the raw documents.
    - Produces a final dense context for answer generation.

The MapReduce pattern prevents the REDUCE phase from being overwhelmed by
K full documents simultaneously. It also enables parallel MAP execution (all
K documents can be summarized concurrently), reducing total wall-clock latency.

"""

'\nRetrieve-Then-Compress (RTC) RAG Pipeline -- Production-Grade\n==============================================================\nArchitecture   : FIVE configurations covering the complete Retrieve-Then-Compress\n                 design space as established by the 2023-2025 research literature:\n                 (1) Baseline RTC                 -- raw retrieval, no compression\n                 (2) RECOMP Extractive            -- sentence-level selection across\n                                                    all retrieved docs jointly\n                 (3) RECOMP Abstractive           -- multi-doc fusion summarization\n                                                    into a single compressed context\n                 (4) LLMLingua Token-Budget RTC   -- perplexity-guided token pruning\n                                                    with hard token budget enforcement\n                 (5) MapReduce + Evidentiality    -- map: per-doc extractive summaries,\n                   

In [2]:
import os
import re
import sys
import time
import pickle
import logging
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from pydantic import BaseModel, Field
import numpy as np

In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS 
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers import EnsembleRetriever , ContextualCompressionRetriever
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter, LLMChainExtractor, LLMChainFilter, LLMListwiseRerank, DocumentCompressorPipeline
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_ollama import ChatOllama
from langchain_community.document_compressors import LLMLinguaCompressor

from langchain_core.output_parsers import BaseOutputParser, StrOutputParser

C:\Users\dhanu\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0521 18:51:23.073000 21408 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:
# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("rtc_rag")


In [5]:

# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class RTCConfig:
    """
    Centralized configuration for the Retrieve-Then-Compress RAG pipeline.

    TOKEN_BUDGET (512):
        The LLMLingua budget controller enforces this as the target number of
        tokens in the compressed context. Tokens = chars / 4 (rough approximation
        for English text with tiktoken cl100k_base encoding).
        512 tokens = ~2,048 characters of compressed context.
        Original: 10 chunks * 450 chars = 4,500 chars (~1,125 tokens).
        Compression ratio target: 1125 / 512 = ~2.2x.
        For aggressive compression, use TOKEN_BUDGET=256 (4.4x ratio).

    LLMLINGUA_COMPRESSION_RATE (0.5):
        Target compression rate for LLMLingua: retain 50% of tokens.
        At 0.5, LLMLingua drops every token where the small model's perplexity
        is below the computed threshold, targeting 50% token retention.
        Valid range: 0.1 (very aggressive, 10% retention) to 0.9 (gentle, 90% retention).
        Empirical finding: LLMLingua can achieve 20x compression (5% retention)
        with only 1.5% performance loss on reasoning tasks. For RAG pipelines,
        0.3-0.5 retention is the production-validated operating range.

    RECOMP_ABSTRACTIVE_MAX_CHARS (800):
        Maximum character length for the abstractive summary (Config 3).
        The abstractive compressor synthesizes all retrieved documents into one
        paragraph. Enforcing a max length prevents the summarizer from producing
        a near-verbatim copy of the original context (which would defeat the purpose
        of compression). 800 chars = ~200 tokens = ~78% reduction from raw 1125 tokens.

    MAPREDUCE_MAP_MAX_CHARS (150):
        Each MAP phase summary is limited to 150 characters per document.
        At K=10 documents, the REDUCE input is max 10*150=1,500 chars.
        The REDUCE phase then compresses to the final answer context.
        MAP max chars should be 1/3 of the raw chunk size: 150/450 = 33% retention per doc.

    EVIDENTIALITY_ITERATIONS (3):
        The evidentiality-guided loop (Config 5) checks whether the compressed
        context is sufficient to answer the query. If not, it retrieves additional
        documents and re-runs the compression. Maximum 3 iterations prevents
        infinite loops when the corpus genuinely lacks the answer.
        Inspired by ECoRAG (2025): an evidentiality evaluator determines whether
        the compressed context C is evidential. If not, more evidence is added
        iteratively until the evaluator judges C as evidential.

    LLM_COMPRESS_TEMPERATURE (0.0):
        All compression LLM calls use temperature=0.0. Compression is a
        FAITHFULNESS task: the output must accurately represent the source content.
        Temperature > 0 introduces variation that risks hallucinating content
        not present in the source documents, which is the primary failure mode
        of abstractive compression.
    """

    # --- Dataset -----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",     "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers",  "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",   "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"

    # --- FAISS -----------------------------------------------------------
    FAISS_PERSIST_DIR: str = "./faiss_rtc_index"

    # --- BM25 ------------------------------------------------------------
    BM25_PERSIST_DIR: str  = "./bm25_rtc_index"
    BM25_PARAMS:      Dict = {"k1": 1.5, "b": 0.75, "delta": 0.5}

    # --- BGE Embeddings --------------------------------------------------
    BIENCODER_MODEL:             str = "BAAI/bge-large-en-v1.5"
    BIENCODER_DEVICE:            str = "cpu"
    BIENCODER_QUERY_INSTRUCTION: str = (
        "Represent this sentence for searching relevant passages: "
    )

    # --- Retrieval K Parameters ------------------------------------------
    BASE_RETRIEVE_K: int = 10   # raw documents retrieved before compression
    BM25_K:          int = 10
    DENSE_K:         int = 10
    FINAL_K:         int = 4    # documents used if no compression (baseline)

    # --- Hybrid RRF Weights ----------------------------------------------
    HYBRID_WEIGHTS: List[float] = [0.4, 0.6]  # [bm25, dense]

    # --- Compression Parameters ------------------------------------------
    TOKEN_BUDGET:                  int   = 512   # LLMLingua token budget
    LLMLINGUA_COMPRESSION_RATE:    float = 0.5   # 50% token retention
    LLMLINGUA_MODEL:               str   = "openai-community/gpt2"  # small local LM
    RECOMP_EXTRACTIVE_MAX_SENTS:   int   = 8     # max retained sentences (extractive)
    RECOMP_ABSTRACTIVE_MAX_CHARS:  int   = 800   # max chars in abstractive summary
    MAPREDUCE_MAP_MAX_CHARS:       int   = 150   # max chars per MAP document summary
    EVIDENTIALITY_ITERATIONS:      int   = 3     # max evidentiality loop iterations
    EVIDENTIALITY_EXTRA_K:         int   = 5     # additional docs per evidentiality iter

    # --- LLM Temperature -------------------------------------------------
    LLM_COMPRESS_TEMPERATURE: float = 0.0
    LLM_ANSWER_TEMPERATURE:   float = 0.0

    # --- Text Splitting (index-time) ------------------------------------
    CHUNK_SIZE:    int = 450
    CHUNK_OVERLAP: int = 60

    # --- Azure OpenAI LLM -----------------------------------------------
    OLLAMA_BASE_URL: str = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
    OLLAMA_MODEL:    str = os.environ.get("OLLAMA_MODEL",    "qwen2.5-coder:7b")
    LLM_MAX_TOKENS: int            = 1024

    # --- Prompts ---------------------------------------------------------
    RAG_PROMPT: str = """You are a precise technical research assistant.
Answer the question using ONLY the information in the compressed context below.
If the answer is not present, respond:
"The provided documents do not contain sufficient information to answer this question."

Compressed context:
{context}

Question: {question}

Precise technical answer:"""

    RECOMP_EXTRACTIVE_PROMPT: str = """You are an extractive sentence selector for a RAG pipeline.
Given a question and a set of sentences from retrieved documents, select ONLY the sentences
that are DIRECTLY RELEVANT to answering the question.

Rules:
1. Output only verbatim sentences from the input. Do NOT paraphrase or modify any sentence.
2. Select at most {max_sents} sentences total across all documents.
3. If no sentences are relevant, output: EMPTY
4. Separate selected sentences with a single newline.
5. Order selected sentences from most to least directly relevant.

Question: {question}

Retrieved sentences:
{sentences}

Selected verbatim sentences (most relevant first):"""

    RECOMP_ABSTRACTIVE_PROMPT: str = """You are a multi-document abstractive summarizer for RAG.
Given a question and multiple retrieved document passages, generate a SINGLE COMPRESSED PARAGRAPH
that synthesizes all information needed to answer the question.

Rules:
1. Maximum {max_chars} characters in the output paragraph.
2. Include all specific facts, numbers, names, and technical details relevant to the question.
3. If retrieved documents contain no relevant information, output: EMPTY
4. Do NOT include general background; focus only on query-specific content.
5. Write in dense, information-rich prose -- no wasted words.

Question: {question}

Retrieved documents ({n_docs} total):
{documents}

Compressed synthesis paragraph (max {max_chars} chars):"""

    MAP_PROMPT: str = """Extract and compress the key information from this document
that is relevant to answering the question.
Maximum {max_chars} characters. Be extremely concise. If nothing is relevant, output: SKIP

Question: {question}

Document ({paper} p{page}):
{content}

Compressed relevant content (max {max_chars} chars):"""

    REDUCE_PROMPT: str = """You are synthesizing per-document summaries into a single coherent context.
Given a question and compressed summaries from multiple documents, produce a final synthesis
that contains all unique information needed to answer the question.
Eliminate redundancy. Preserve all specific facts and numbers.
Maximum {max_chars} total characters.

Question: {question}

Per-document summaries:
{summaries}

Final synthesized context (max {max_chars} chars):"""

    EVIDENTIALITY_PROMPT: str = """You are an evidentiality evaluator for a RAG pipeline.
Given a question and a compressed context, determine whether the context contains
SUFFICIENT EVIDENCE to answer the question accurately and specifically.

Requirements for "sufficient":
- The context must contain the specific facts, numbers, or mechanisms needed for the answer.
- Vague or general information does NOT count as sufficient.
- If the answer would require inference beyond what is stated, it is NOT sufficient.

Output ONLY one word: SUFFICIENT or INSUFFICIENT

Question: {question}

Compressed context:
{context}

Evidentiality verdict (SUFFICIENT or INSUFFICIENT):"""



In [6]:

# ===========================================================================
# SECTION 2 -- RTC TRACE DATA CLASS
# ===========================================================================

@dataclass
class RTCTrace:
    """
    Records the complete Retrieve-Then-Compress execution trace.

    compression_ratio: the primary production metric.
        = raw_chars_total / compressed_chars_total
        Target: >= 3.0 for meaningful context reduction.
        If ratio < 1.5, compression is not effective -- investigate threshold tuning.

    evidentiality_iterations: for Config 5, the number of retrieve-compress-check
        cycles before the evidentiality evaluator judges the context sufficient.
        Typical: 1-2 iterations. If consistently 3 (max), the corpus lacks the answer.

    llm_compress_calls: LLM API calls consumed by the compression step alone
        (excluding the final answer generation call). Critical for cost accounting:
        compression_cost = llm_compress_calls * cost_per_llm_call.

    selective_augmentation_triggered: True if the compressor decided the retrieved
        documents were irrelevant and produced an EMPTY output, triggering fallback.
        Monitoring this rate detects retrieval quality issues (high selective aug
        rate = retrieval is not finding relevant content for the query set).
    """
    question:                          str
    strategy:                          str
    raw_docs:                          List[Document] = field(default_factory=list)
    compressed_context:                str            = ""
    raw_chars_total:                   int            = 0
    compressed_chars_total:            int            = 0
    compression_ratio:                 float          = 1.0
    llm_compress_calls:                int            = 0
    selective_augmentation_triggered:  bool           = False
    evidentiality_iterations:          int            = 0
    final_answer:                      str            = ""
    retrieval_ms:                      float          = 0.0
    compression_ms:                    float          = 0.0
    generation_ms:                     float          = 0.0
    total_ms:                          float          = 0.0

    def compute_ratios(self) -> None:
        self.raw_chars_total       = sum(len(d.page_content) for d in self.raw_docs)
        self.compressed_chars_total = len(self.compressed_context)
        self.compression_ratio     = self.raw_chars_total / max(self.compressed_chars_total, 1)

    def print_trace(self) -> None:
        print(f"\n{'='*74}")
        print(f"TRACE: {self.strategy}")
        print(f"Query: {self.question[:80]}")
        print(f"{'='*74}")
        print(f"\n  Raw docs: {len(self.raw_docs)}  ({self.raw_chars_total:,} chars)")
        print(f"  Compressed context: {self.compressed_chars_total:,} chars")
        print(f"  Compression ratio: {self.compression_ratio:.2f}x")
        print(f"  LLM compress calls: {self.llm_compress_calls}")
        print(f"  Selective aug triggered: {self.selective_augmentation_triggered}")
        if self.evidentiality_iterations:
            print(f"  Evidentiality iterations: {self.evidentiality_iterations}")
        print(f"\n  Compressed context preview (first 300 chars):")
        print(f"  \"{self.compressed_context[:300]}...\"")
        print(f"\n  Latency: retrieval={self.retrieval_ms:.0f}ms  "
              f"compress={self.compression_ms:.0f}ms  "
              f"gen={self.generation_ms:.0f}ms  "
              f"total={self.total_ms:.0f}ms")
        print(f"\n  ANSWER:\n  {self.final_answer[:450]}")
        print("=" * 74 + "\n")


In [7]:

# ===========================================================================
# SECTION 3 -- PDF DOWNLOADER AND CHUNKER
# ===========================================================================

def download_pdfs(config: RTCConfig) -> List[Path]:
    """Download research PDFs with file-existence caching."""
    dl_dir = Path(config.PDF_DOWNLOAD_DIR)
    dl_dir.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []
    for name, url in config.PDF_SOURCES:
        dest = dl_dir / f"{name}.pdf"
        if dest.exists() and dest.stat().st_size > 10_000:
            logger.info("Cached: %s  (%.1f KB)", dest.name, dest.stat().st_size / 1024)
            paths.append(dest)
            continue
        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "Mozilla/5.0 (RAG-RTC-Pipeline/1.0)"}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            if len(data) < 10_000:
                raise RuntimeError(f"File too small: {url}")
            dest.write_bytes(data)
            logger.info("Saved: %s  (%.1f KB)", dest.name, len(data) / 1024)
            paths.append(dest)
            time.sleep(1.0)
        except Exception as exc:
            raise RuntimeError(
                f"Cannot download '{url}'. Place manually at '{dest}'."
            ) from exc
    return paths


def load_and_chunk_documents(
    pdf_paths: List[Path],
    config: RTCConfig,
) -> List[Document]:
    """Load PDFs and split into chunks with paper_name and page metadata."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
        add_start_index=True,
    )
    all_chunks: List[Document] = []
    for pdf_path in pdf_paths:
        paper_name = pdf_path.stem.replace("_", " ").title()
        try:
            pages = PyPDFLoader(str(pdf_path)).load()
        except Exception as exc:
            raise RuntimeError(f"Failed to load '{pdf_path}': {exc}") from exc
        for page in pages:
            page.metadata.update({"source": pdf_path.name, "paper_name": paper_name})
        chunks = splitter.split_documents(pages)
        logger.info("  %s -> %d pages -> %d chunks", pdf_path.name, len(pages), len(chunks))
        all_chunks.extend(chunks)
    logger.info("Total chunks: %d", len(all_chunks))
    return all_chunks



In [30]:


# ===========================================================================
# SECTION 4 -- INDEX AND MODEL BUILDERS
# ===========================================================================

def bm25_preprocess(text: str) -> List[str]:
    return text.lower().split()


def build_bge_embeddings(config: RTCConfig) -> HuggingFaceEmbeddings:
    logger.info("Loading BGE: %s", config.BIENCODER_MODEL)
    return HuggingFaceEmbeddings(
        model_name=config.BIENCODER_MODEL,
        model_kwargs={"device": config.BIENCODER_DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )


def build_faiss_index(
    chunks: List[Document],
    embeddings: HuggingFaceEmbeddings,
    config: RTCConfig,
 ) -> FAISS:
    faiss_file = Path(config.FAISS_PERSIST_DIR) / "index.faiss"
    if faiss_file.exists():
        logger.info("Loading FAISS from '%s' ...", config.FAISS_PERSIST_DIR)
        try:
            vs = FAISS.load_local(
                config.FAISS_PERSIST_DIR, embeddings,
                allow_dangerous_deserialization=True,
            )
            logger.info("FAISS loaded. Vectors: %d", vs.index.ntotal)
            return vs
        except Exception as exc:
            logger.warning("FAISS load failed (%s), rebuilding ...", exc)
    logger.info("Building FAISS from %d chunks ...", len(chunks))
    t0 = time.perf_counter()
    vs = FAISS.from_documents(chunks, embeddings)
    logger.info("FAISS built. Vectors: %d  (%.2fs)", vs.index.ntotal, time.perf_counter() - t0)
    Path(config.FAISS_PERSIST_DIR).mkdir(parents=True, exist_ok=True)
    vs.save_local(config.FAISS_PERSIST_DIR)
    return vs


def build_bm25_index(chunks: List[Document], config: RTCConfig) -> BM25Retriever:
    cache = Path(config.BM25_PERSIST_DIR) / "bm25plus.pkl"
    if cache.exists():
        logger.info("Loading BM25 from '%s' ...", cache)
        try:
            with open(cache, "rb") as f:
                state = pickle.load(f)
            return BM25Retriever(
                vectorizer=state["vectorizer"], docs=state["docs"],
                k=state["k"], preprocess_func=state["preprocess_func"],
            )
        except Exception as exc:
            logger.warning("BM25 cache invalid (%s). Rebuilding and overwriting cache.", exc)

    logger.info("Building BM25Plus from %d chunks ...", len(chunks))
    texts = [d.page_content for d in chunks]
    tokenized_texts = [bm25_preprocess(t) for t in texts]

    # Some langchain_community versions route through BM25Okapi even when
    # bm25_variant='plus', which crashes on the BM25Plus-only 'delta' parameter.
    # Build BM25Plus directly from rank_bm25 to guarantee compatibility.
    try:
        from rank_bm25 import BM25Plus
        vectorizer = BM25Plus(tokenized_texts, **config.BM25_PARAMS)
    except Exception as exc:
        logger.warning("BM25Plus unavailable (%s). Falling back to BM25Okapi.", exc)
        from rank_bm25 import BM25Okapi
        okapi_params = {k: v for k, v in config.BM25_PARAMS.items() if k != "delta"}
        vectorizer = BM25Okapi(tokenized_texts, **okapi_params)

    ret = BM25Retriever(
        vectorizer=vectorizer,
        docs=chunks,
        k=config.BM25_K,
        preprocess_func=bm25_preprocess,
    )

    cache.parent.mkdir(parents=True, exist_ok=True)
    with open(cache, "wb") as f:
        pickle.dump(
            {
                "vectorizer": ret.vectorizer,
                "docs": ret.docs,
                "k": ret.k,
                "preprocess_func": ret.preprocess_func,
            },
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
    return ret


def build_ollama_llm(config: RTCConfig, temperature: float = None) -> ChatOllama:
    """Connect to local Ollama and return a ChatOllama instance."""
    import urllib.request, urllib.error
    base_url = getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434')
    model    = getattr(config, 'OLLAMA_MODEL',    'qwen2.5-coder:7b')
    try:
        urllib.request.urlopen(base_url, timeout=3)
    except Exception as exc:
        raise RuntimeError(
            f"Ollama not reachable at {base_url}. Start Ollama and run: "
            f"ollama pull qwen2.5-coder:7b"
        ) from exc
    return ChatOllama(
        base_url=base_url,
        model=model,
        temperature=getattr(config, 'LLM_TEMPERATURE', temperature),
        num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
    )
def build_hybrid_retriever(
    vectorstore: FAISS,
    bm25: BM25Retriever,
    config: RTCConfig,
 ) -> EnsembleRetriever:
    bm25_r  = BM25Retriever(
        vectorizer=bm25.vectorizer, docs=bm25.docs,
        k=config.BM25_K, preprocess_func=bm25.preprocess_func,
    )
    dense_r = vectorstore.as_retriever(
        search_type="similarity", search_kwargs={"k": config.DENSE_K},
    )
    return EnsembleRetriever(
        retrievers=[bm25_r, dense_r],
        weights=config.HYBRID_WEIGHTS,
    )

In [9]:

# ===========================================================================
# SECTION 5 -- SHARED UTILITIES
# ===========================================================================

def retrieve_raw(
    question:  str,
    retriever: EnsembleRetriever,
    config:    RTCConfig,
) -> Tuple[List[Document], float]:
    t0   = time.perf_counter()
    docs = retriever.invoke(question)[: config.BASE_RETRIEVE_K]
    return docs, (time.perf_counter() - t0) * 1000


def generate_answer(
    question:           str,
    compressed_context: str,
    llm:                ChatOllama,
    config:             RTCConfig,
) -> Tuple[str, float]:
    prompt = ChatPromptTemplate.from_template(config.RAG_PROMPT)
    t0     = time.perf_counter()
    answer = (prompt | llm | StrOutputParser()).invoke(
        {"context": compressed_context, "question": question}
    )
    return answer, (time.perf_counter() - t0) * 1000


def extract_sentences(docs: List[Document]) -> List[Tuple[str, str, int]]:
    """
    Split documents into sentences for RECOMP extractive compression.

    Returns: List of (sentence, paper_name, page) tuples.
    Splitting by period+space is a pragmatic approximation. For production:
    use spaCy or NLTK sentence tokenizers which handle abbreviations (e.g.,
    "Fig. 2" is not a sentence boundary) and nested punctuation correctly.
    """
    sentences = []
    for doc in docs:
        paper = doc.metadata.get("paper_name", "Unknown")
        page  = doc.metadata.get("page", "?")
        raw   = doc.page_content.strip()
        parts = re.split(r'(?<=[.!?])\s+', raw)
        for part in parts:
            part = part.strip()
            if len(part) > 20:
                sentences.append((part, paper, page))
    return sentences



In [10]:

# ===========================================================================
# SECTION 6 -- FIVE RTC CONFIGURATIONS
# ===========================================================================

# ---------------------------------------------------------------------------
# CONFIG 1: Baseline RTC -- raw retrieval, no compression
# ---------------------------------------------------------------------------

def run_config1_baseline(
    question:    str,
    vectorstore: FAISS,
    bm25:        BM25Retriever,
    llm_answer:  ChatOllama,
    config:      RTCConfig,
) -> RTCTrace:
    """
    Configuration 1: Baseline (no compression).

    Raw hybrid retrieval, top-FINAL_K documents concatenated verbatim.
    Control condition: establishes token budget and answer quality floor.
    """
    trace = RTCTrace(question=question, strategy="Config1_Baseline_NoCompression")
    t0    = time.perf_counter()

    retriever                = build_hybrid_retriever(vectorstore, bm25, config)
    raw_docs, trace.retrieval_ms = retrieve_raw(question, retriever, config)
    trace.raw_docs           = raw_docs

    # Baseline: no compression, top-FINAL_K docs concatenated
    context_docs             = raw_docs[: config.FINAL_K]
    trace.compressed_context = "\n\n---\n\n".join(
        f"[{d.metadata.get('paper_name','?')} p{d.metadata.get('page','?')}]\n"
        f"{d.page_content.strip()}"
        for d in context_docs
    )
    trace.compute_ratios()

    answer, trace.generation_ms = generate_answer(
        question, trace.compressed_context, llm_answer, config
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [11]:

# ---------------------------------------------------------------------------
# CONFIG 2: RECOMP Extractive
# ---------------------------------------------------------------------------

def run_config2_recomp_extractive(
    question:     str,
    vectorstore:  FAISS,
    bm25:         BM25Retriever,
    llm_compress: ChatOllama,
    llm_answer:   ChatOllama,
    config:       RTCConfig,
) -> RTCTrace:
    """
    Configuration 2: RECOMP Extractive Compression (zero-shot LLM implementation).

    Source: Xu et al., ICLR 2024, "RECOMP: Improving Retrieval-Augmented LMs
    with Compression and Selective Augmentation"

    The RECOMP extractive compressor filters out irrelevant sentences, retaining
    only the most pertinent ones from the retrieved document set. Contrastive
    learning is employed to train the extractive compressor enabling it to select
    key sentences effectively.

    This implementation uses GPT-4o in zero-shot mode to perform the same role
    as a trained extractive compressor. The key difference from LLMChainExtractor:
        - LLMChainExtractor: processes each document INDEPENDENTLY in a separate call.
        - RECOMP Extractive: processes ALL retrieved sentences JOINTLY in a single call.

    The JOINT processing is architecturally important:
        - When the same information appears in multiple documents (chunking overlap),
          RECOMP's joint extractor sees the redundancy and selects only ONE copy.
        - LLMChainExtractor processes independently and may select the same sentence
          from two overlapping chunks, wasting context budget on duplicates.
        - RECOMP's joint view enables cross-document relevance comparison:
          "Sentence A from doc 3 is more directly relevant than sentence B from doc 1."

    SELECTIVE AUGMENTATION: If the LLM determines no sentences are relevant
    (outputs "EMPTY"), this signals that retrieval failed to find relevant content.
    The pipeline detects this and falls back to the raw top-2 documents.
    Monitoring selective_augmentation_triggered in production reveals query types
    that the retrieval stage consistently fails on.

    Args:
        question      : User query.
        vectorstore   : FAISS index.
        bm25          : BM25Plus retriever.
        llm_compress  : ChatOllama (temperature=0.0, extractive selection).
        llm_answer    : ChatOllama (temperature=0.0, answer generation).
        config        : RTCConfig.

    Returns:
        RTCTrace with sentence-level extraction metrics.
    """
    trace = RTCTrace(question=question, strategy="Config2_RECOMP_Extractive")
    t0    = time.perf_counter()

    retriever                = build_hybrid_retriever(vectorstore, bm25, config)
    raw_docs, trace.retrieval_ms = retrieve_raw(question, retriever, config)
    trace.raw_docs           = raw_docs

    # Extract all sentences from all retrieved documents jointly
    all_sentences = extract_sentences(raw_docs)
    logger.info("RECOMP Extractive: %d sentences from %d docs", len(all_sentences), len(raw_docs))

    # Format sentences with source labels for the prompt
    sentences_text = "\n".join([
        f"[{paper[:25]} p{page}] {sent}"
        for sent, paper, page in all_sentences
    ])

    t_compress   = time.perf_counter()
    ext_prompt   = ChatPromptTemplate.from_template(config.RECOMP_EXTRACTIVE_PROMPT)
    compressed   = (ext_prompt | llm_compress | StrOutputParser()).invoke({
        "question":  question,
        "sentences": sentences_text[:4000],   # safety: cap input to LLM at 4000 chars
        "max_sents": config.RECOMP_EXTRACTIVE_MAX_SENTS,
    }).strip()
    trace.compression_ms    = (time.perf_counter() - t_compress) * 1000
    trace.llm_compress_calls = 1

    # Detect EMPTY (selective augmentation) output
    if not compressed or compressed.upper().startswith("EMPTY"):
        logger.warning("RECOMP Extractive: EMPTY output (selective augmentation). Fallback.")
        trace.selective_augmentation_triggered = True
        compressed = "\n".join([d.page_content[:200] for d in raw_docs[:2]])

    trace.compressed_context = compressed
    trace.compute_ratios()
    logger.info(
        "RECOMP Extractive: %d raw chars -> %d compressed  (%.2fx)",
        trace.raw_chars_total, trace.compressed_chars_total, trace.compression_ratio,
    )

    answer, trace.generation_ms = generate_answer(question, compressed, llm_answer, config)
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [12]:

# ---------------------------------------------------------------------------
# CONFIG 3: RECOMP Abstractive
# ---------------------------------------------------------------------------

def run_config3_recomp_abstractive(
    question:     str,
    vectorstore:  FAISS,
    bm25:         BM25Retriever,
    llm_compress: ChatOllama,
    llm_answer:   ChatOllama,
    config:       RTCConfig,
) -> RTCTrace:
    """
    Configuration 3: RECOMP Abstractive Compression (multi-document fusion).

    Source: Xu et al., ICLR 2024 -- the abstractive compressor component.

    The abstractive compressor generates a summary by fusing information from
    multiple retrieved documents. The abstractive compressor is distilled from a
    large language model (GPT-3 or GPT-4), achieving strong summarization performance.

    Key differences from RECOMP Extractive (Config 2):
        EXTRACTIVE: Output is verbatim source sentences. Source-faithful.
                    Cannot combine information across documents.
                    Cannot restructure information for better coherence.
        ABSTRACTIVE: Output is generated (may paraphrase). Not source-verbatim.
                     CAN synthesize information from multiple documents into one claim.
                     CAN structure the compressed summary for optimal LLM readability.
                     Risk: small probability of hallucinating content not in sources.

    When abstractive compression outperforms extractive:
        - Multi-hop queries: "What attention mechanism does BERT use for pre-training?"
          requires combining: (a) BERT uses transformer architecture; (b) transformer uses
          self-attention; (c) self-attention computes query-key-value products.
          Extractive returns three separate sentences. Abstractive generates:
          "BERT uses multi-head self-attention (scaled dot-product query-key-value) in its
          transformer encoder for pre-training." More information-dense for the same budget.
        - Redundant multi-document sets: when 10 retrieved chunks all say variations of the
          same fact, extractive must pick one, losing the others. Abstractive deduplicates
          and represents the consensus fact once.

    A 2024 study comparing compression methods found abstractive compression at 4.5x ratio
    decreased performance by 4.69 F1 points on 2WikiMultihopQA while extractive at the same
    ratio improved by +7.89 F1 -- illustrating that abstractive compression is best suited
    for factoid queries while extractive is better for multi-hop queries.

    Args:
        question      : User query.
        vectorstore   : FAISS index.
        bm25          : BM25Plus retriever.
        llm_compress  : ChatOllama (temperature=0.0, abstractive summarizer).
        llm_answer    : ChatOllama (temperature=0.0, answer generation).
        config        : RTCConfig.

    Returns:
        RTCTrace with multi-doc fusion metrics.
    """
    trace = RTCTrace(question=question, strategy="Config3_RECOMP_Abstractive")
    t0    = time.perf_counter()

    retriever                = build_hybrid_retriever(vectorstore, bm25, config)
    raw_docs, trace.retrieval_ms = retrieve_raw(question, retriever, config)
    trace.raw_docs           = raw_docs

    # Format all retrieved documents for the abstractive prompt
    docs_text = "\n\n".join([
        f"[Doc {i+1} | {d.metadata.get('paper_name','?')[:25]} p{d.metadata.get('page','?')}]\n"
        f"{d.page_content.strip()[:300]}"
        for i, d in enumerate(raw_docs)
    ])

    t_compress    = time.perf_counter()
    abs_prompt    = ChatPromptTemplate.from_template(config.RECOMP_ABSTRACTIVE_PROMPT)
    compressed    = (abs_prompt | llm_compress | StrOutputParser()).invoke({
        "question":  question,
        "documents": docs_text,
        "n_docs":    len(raw_docs),
        "max_chars": config.RECOMP_ABSTRACTIVE_MAX_CHARS,
    }).strip()
    trace.compression_ms     = (time.perf_counter() - t_compress) * 1000
    trace.llm_compress_calls = 1

    # Detect EMPTY selective augmentation
    if not compressed or compressed.upper().startswith("EMPTY"):
        logger.warning("RECOMP Abstractive: EMPTY output (selective augmentation). Fallback.")
        trace.selective_augmentation_triggered = True
        compressed = raw_docs[0].page_content[:config.RECOMP_ABSTRACTIVE_MAX_CHARS]

    # Enforce max_chars limit in case LLM exceeds it
    compressed = compressed[:config.RECOMP_ABSTRACTIVE_MAX_CHARS]

    trace.compressed_context = compressed
    trace.compute_ratios()
    logger.info(
        "RECOMP Abstractive: %d raw chars -> %d compressed  (%.2fx)",
        trace.raw_chars_total, trace.compressed_chars_total, trace.compression_ratio,
    )

    answer, trace.generation_ms = generate_answer(question, compressed, llm_answer, config)
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [13]:
from langchain_community.document_compressors import LLMLinguaCompressor


In [14]:

# ---------------------------------------------------------------------------
# CONFIG 4: LLMLingua Token-Budget-Aware Compression
# ---------------------------------------------------------------------------

def run_config4_llmlingua(
    question:    str,
    vectorstore: FAISS,
    bm25:        BM25Retriever,
    llm_answer:  ChatOllama,
    config:      RTCConfig,
) -> RTCTrace:
    """
    Configuration 4: LLMLingua Token-Budget-Aware Prompt Compression.

    Source: Jiang et al., EMNLP 2023, "LLMLingua: Compressing Prompts for
    Accelerated Inference of Large Language Models"

    LLMLingua utilizes a compact, well-trained language model (e.g., GPT2-small,
    LLaMA-7B) to identify and remove non-essential tokens in prompts. This approach
    enables efficient inference with large language models (LLMs), achieving up to
    20x compression with minimal performance loss.

    LLMLingua has been integrated into LangChain and LlamaIndex as
    LLMLinguaCompressor, usable directly with ContextualCompressionRetriever.

    Three-stage LLMLingua compression pipeline:
        STAGE 1 (Budget Controller): Estimate the minimum token budget needed to
            preserve semantic integrity. Allocates the target token count across
            the documents in the prompt.
        STAGE 2 (Coarse-Grained): Segment the prompt and estimate each segment's
            perplexity using the small language model. Low-perplexity segments
            (the small model can predict them easily -- they are redundant) receive
            lower token budgets and are compressed more aggressively.
        STAGE 3 (Fine-Grained): Within each segment, drop individual tokens with
            perplexity below the computed threshold, building an iterative token
            classification. This produces the final compressed prompt.

    LLMLingua vs LLM-based compressors (RECOMP):
        LLMLingua:  token-level pruning using a local small model (GPT2-small, ~500MB).
                    Zero LLM API calls during compression.
                    Output is "compressed language" -- individual tokens removed.
                    GPT-4 can understand compressed prompts produced by GPT2.
                    Compression rate controlled by a continuous parameter (0.0-1.0).
        RECOMP:     sentence-level extraction using the answer LLM.
                    Requires LLM API calls during compression.
                    Output is grammatically correct extracted/synthesized text.
                    Compression granularity is at the sentence boundary.

    Token-budget enforcement:
        config.TOKEN_BUDGET = 512 tokens target.
        LLMLinguaCompressor.target_token = 512.
        If the compressed output exceeds the budget, LLMLingua applies more
        aggressive pruning. The budget is a HARD constraint.

    Installation: pip install llmlingua
    LangChain import: from langchain_community.document_compressors import LLMLinguaCompressor

    Args:
        question    : User query.
        vectorstore : FAISS index.
        bm25        : BM25Plus retriever.
        llm_answer  : ChatOllama (temperature=0.0, answer generation).
        config      : RTCConfig.

    Returns:
        RTCTrace with token-budget-aware compression metrics.
    """
    trace = RTCTrace(question=question, strategy="Config4_LLMLingua_TokenBudget")
    t0    = time.perf_counter()

    retriever                = build_hybrid_retriever(vectorstore, bm25, config)
    raw_docs, trace.retrieval_ms = retrieve_raw(question, retriever, config)
    trace.raw_docs           = raw_docs

    t_compress = time.perf_counter()
    try:
        from langchain_community.document_compressors import LLMLinguaCompressor

        compressor = LLMLinguaCompressor(
            model_name=config.LLMLINGUA_MODEL,
            device_map="cpu",
            target_token=config.TOKEN_BUDGET,
            rank_method="longllmlingua",
            additional_compress_ratio=config.LLMLINGUA_COMPRESSION_RATE,
        )
        cc_retriever = ContextualCompressionRetriever(
            base_compressor=compressor,
            base_retriever=build_hybrid_retriever(vectorstore, bm25, config),
        )
        compressed_docs = cc_retriever.invoke(question)
        trace.compression_ms     = (time.perf_counter() - t_compress) * 1000
        trace.llm_compress_calls = 0   # LLMLingua uses local GPT2, zero API calls

        if compressed_docs:
            trace.compressed_context = "\n\n".join([
                d.page_content for d in compressed_docs
            ])
        else:
            logger.warning("LLMLingua: no compressed docs returned. Fallback to raw top-2.")
            trace.selective_augmentation_triggered = True
            trace.compressed_context = "\n".join([
                raw_docs[i].page_content[:200] for i in range(min(2, len(raw_docs)))
            ])

    except ImportError:
        logger.warning(
            "LLMLingua not installed. Run: pip install llmlingua\n"
            "Falling back to manual token-budget truncation."
        )
        # Manual fallback: truncate combined context to TOKEN_BUDGET chars * 4 chars/token
        char_budget = config.TOKEN_BUDGET * 4
        combined    = "\n\n".join([d.page_content for d in raw_docs])
        trace.compression_ms     = (time.perf_counter() - t_compress) * 1000
        trace.llm_compress_calls = 0
        trace.compressed_context = combined[:char_budget]

    trace.compute_ratios()
    logger.info(
        "LLMLingua: %d raw chars -> %d compressed  (%.2fx, budget=%d tokens)",
        trace.raw_chars_total, trace.compressed_chars_total,
        trace.compression_ratio, config.TOKEN_BUDGET,
    )

    answer, trace.generation_ms = generate_answer(
        question, trace.compressed_context, llm_answer, config
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace


In [15]:

# ---------------------------------------------------------------------------
# CONFIG 5: MapReduce + Evidentiality-Guided Iterative RTC [BEST]
# ---------------------------------------------------------------------------

def run_config5_mapreduce_evidentiality(
    question:     str,
    vectorstore:  FAISS,
    bm25:         BM25Retriever,
    llm_compress: ChatOllama,
    llm_answer:   ChatOllama,
    config:       RTCConfig,
) -> RTCTrace:
    """
    Configuration 5: MapReduce + Evidentiality-Guided Iterative RTC [BEST].

    This configuration combines two complementary compression innovations:

    PART A -- MapReduce RAG Compression:
        Recent innovation in context management includes MapReduce-inspired
        approaches where the LLM first processes individual chunks (MAP), then
        synthesizes these intermediate outputs into a final response (REDUCE).
        This technique has shown promising results for complex queries requiring
        information from many documents.

        MAP Phase (parallel, K calls):
            For each retrieved document independently:
                Prompt: "Extract and compress the key information relevant to the query.
                         Maximum {max_chars} characters. If nothing is relevant: SKIP."
            Output: K compressed summaries (some SKIP'd).
            Advantage: Each MAP call sees only ONE document -> smaller context, faster.
            Parallelization: all K MAP calls are independent -> asyncio.gather() in production.

        REDUCE Phase (1 call):
            Input: all non-SKIP'd MAP summaries concatenated.
            Prompt: "Synthesize into a final coherent context, eliminating redundancy."
            Output: a single dense context for the answer generation LLM.
            Advantage: The REDUCE LLM sees COMPRESSED intermediates (not raw 450-char chunks).
            The REDUCE input is at most K * MAP_MAX_CHARS = 10 * 150 = 1,500 chars.

    PART B -- Evidentiality-Guided Iterative Loop:
        Inspired by ECoRAG (2025): the evidentiality-guided compressor compresses
        retrieved documents and the evidentiality evaluator determines whether the
        compressed context C is evidential. If not, more evidence is added iteratively
        until the evaluator judges C as evidential.

        The evidentiality loop:
            iter 0: Retrieve BASE_RETRIEVE_K docs, run MAP-REDUCE, check evidentiality.
            If INSUFFICIENT: retrieve EVIDENTIALITY_EXTRA_K=5 additional docs.
            iter 1: Run MAP-REDUCE on BASE_RETRIEVE_K+5 docs, check evidentiality.
            Repeat until SUFFICIENT or EVIDENTIALITY_ITERATIONS=3 reached.

        Evidentiality check prompt:
            "Does this compressed context contain SUFFICIENT EVIDENCE to answer
             the question accurately and specifically?
             Output: SUFFICIENT or INSUFFICIENT."

        WHY EVIDENTIALITY IS THE CORRECT TERMINATION CRITERION:
            Simply retrieving more documents is not guaranteed to improve compression
            quality. The evidentiality check tests whether the COMPRESSED CONTEXT --
            after MAP-REDUCE compression -- actually contains the answer.
            It is possible that 10 retrieved documents compress to a context with
            INSUFFICIENT evidence while 15 documents compress to SUFFICIENT evidence,
            not because the 5 new documents are individually more relevant (they may
            not be by bi-encoder score) but because they contain the specific sentence
            that completes the factual chain needed for the answer.

    COMBINED RESULT [BEST]:
        MapReduce achieves parallelizable compression with clear REDUCE quality control.
        Evidentiality loop ensures the compressed context actually answers the query.
        LLM calls: MAP (K) + REDUCE (1) + evidentiality_check (1) + possible iterations.
        Typical total: 12-14 LLM calls for K=10, 1 evidentiality check, 1 answer.
        Highest answer quality configuration.

    Args:
        question      : User query.
        vectorstore   : FAISS index.
        bm25          : BM25Plus retriever.
        llm_compress  : ChatOllama for MAP, REDUCE, and evidentiality calls.
        llm_answer    : ChatOllama for final answer generation.
        config        : RTCConfig.

    Returns:
        RTCTrace with MapReduce metrics and evidentiality iteration count.
    """
    trace = RTCTrace(
        question=question, strategy="Config5_MapReduce_Evidentiality [BEST]"
    )
    t0 = time.perf_counter()

    retriever = build_hybrid_retriever(vectorstore, bm25, config)
    t_ret     = time.perf_counter()
    all_docs  = retriever.invoke(question)[: config.BASE_RETRIEVE_K]
    trace.retrieval_ms = (time.perf_counter() - t_ret) * 1000
    trace.raw_docs     = all_docs

    map_prompt    = ChatPromptTemplate.from_template(config.MAP_PROMPT)
    reduce_prompt = ChatPromptTemplate.from_template(config.REDUCE_PROMPT)
    evid_prompt   = ChatPromptTemplate.from_template(config.EVIDENTIALITY_PROMPT)
    n_compress_calls = 0

    t_compress = time.perf_counter()

    for iteration in range(config.EVIDENTIALITY_ITERATIONS):
        # ---- MAP Phase ------------------------------------------------
        map_summaries: List[str] = []

        for doc in all_docs:
            summary = (map_prompt | llm_compress | StrOutputParser()).invoke({
                "question": question,
                "paper":    doc.metadata.get("paper_name", "?")[:25],
                "page":     doc.metadata.get("page", "?"),
                "content":  doc.page_content[:350],
                "max_chars": config.MAPREDUCE_MAP_MAX_CHARS,
            }).strip()
            n_compress_calls += 1

            if not summary.upper().startswith("SKIP") and len(summary) > 10:
                map_summaries.append(summary)

        logger.info(
            "Config5 iter %d MAP: %d/%d docs produced non-SKIP summaries",
            iteration, len(map_summaries), len(all_docs),
        )

        if not map_summaries:
            logger.warning("Config5: all MAP summaries SKIP'd. Using raw top-2 content.")
            final_context = "\n".join([d.page_content[:200] for d in all_docs[:2]])
            break

        # ---- REDUCE Phase ---------------------------------------------
        summaries_text = "\n\n".join([
            f"[Summary {i+1}]\n{s}" for i, s in enumerate(map_summaries)
        ])
        max_reduce_chars = config.RECOMP_ABSTRACTIVE_MAX_CHARS

        final_context = (reduce_prompt | llm_compress | StrOutputParser()).invoke({
            "question":  question,
            "summaries": summaries_text,
            "max_chars": max_reduce_chars,
        }).strip()[:max_reduce_chars]
        n_compress_calls += 1

        logger.info(
            "Config5 iter %d REDUCE: %d summaries -> %d chars",
            iteration, len(map_summaries), len(final_context),
        )

        # ---- Evidentiality Check -------------------------------------
        evidentiality = (evid_prompt | llm_compress | StrOutputParser()).invoke({
            "question": question,
            "context":  final_context,
        }).strip().upper()
        n_compress_calls += 1
        trace.evidentiality_iterations = iteration + 1

        logger.info("Config5 iter %d evidentiality: %s", iteration, evidentiality)

        if evidentiality.startswith("SUFFICIENT"):
            logger.info("Config5: SUFFICIENT evidence at iteration %d.", iteration)
            break

        # ---- Expand Retrieval (insufficient evidence) ----------------
        if iteration < config.EVIDENTIALITY_ITERATIONS - 1:
            logger.info(
                "Config5: INSUFFICIENT. Retrieving %d additional docs.",
                config.EVIDENTIALITY_EXTRA_K,
            )
            extra_docs = retriever.invoke(question)[
                len(all_docs): len(all_docs) + config.EVIDENTIALITY_EXTRA_K
            ]
            all_docs = all_docs + extra_docs
            trace.raw_docs = all_docs
        else:
            logger.warning("Config5: max iterations reached. Using final compressed context.")

    trace.compression_ms     = (time.perf_counter() - t_compress) * 1000
    trace.llm_compress_calls = n_compress_calls
    trace.compressed_context = final_context
    trace.compute_ratios()

    logger.info(
        "Config5 MapReduce+Evid: %d raw chars -> %d compressed  "
        "(%.2fx, %d LLM compress calls, %d evid iters)",
        trace.raw_chars_total, trace.compressed_chars_total,
        trace.compression_ratio, n_compress_calls, trace.evidentiality_iterations,
    )

    answer, trace.generation_ms = generate_answer(
        question, trace.compressed_context, llm_answer, config
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [16]:

# ===========================================================================
# SECTION 7 -- COMPARATIVE RUNNER
# ===========================================================================

def run_all_configs(
    question:     str,
    vectorstore:  FAISS,
    bm25:         BM25Retriever,
    llm_compress: ChatOllama,
    llm_answer:   ChatOllama,
    config:       RTCConfig,
) -> Dict[str, RTCTrace]:
    print("\n" + "#" * 78)
    print(f"QUERY: {question}")
    print("#" * 78)

    runners = {
        "Config1_Baseline": lambda q: run_config1_baseline(
            q, vectorstore, bm25, llm_answer, config),
        "Config2_RECOMP_Extractive": lambda q: run_config2_recomp_extractive(
            q, vectorstore, bm25, llm_compress, llm_answer, config),
        "Config3_RECOMP_Abstractive": lambda q: run_config3_recomp_abstractive(
            q, vectorstore, bm25, llm_compress, llm_answer, config),
        "Config4_LLMLingua_TokenBudget": lambda q: run_config4_llmlingua(
            q, vectorstore, bm25, llm_answer, config),
        "Config5_MapReduce_Evidentiality [BEST]": lambda q: run_config5_mapreduce_evidentiality(
            q, vectorstore, bm25, llm_compress, llm_answer, config),
    }

    traces: Dict[str, RTCTrace] = {}

    for label, fn in runners.items():
        print(f"\n{'='*62}\nRUNNING: {label}\n{'='*62}")
        try:
            trace = fn(question)
            trace.print_trace()
            traces[label] = trace
        except Exception as exc:
            logger.error("Config %s failed: %s", label, exc, exc_info=True)

    # Summary
    print("\n" + "=" * 78)
    print("RETRIEVE-THEN-COMPRESS SUMMARY")
    print(f"{'Config':<48} {'Ratio':>6} {'LLMcmp':>7} {'Comp(ms)':>9} {'Total(ms)':>10}")
    print("-" * 78)
    for lbl, tr in traces.items():
        print(
            f"{lbl:<48} {tr.compression_ratio:>6.1f}x "
            f"{tr.llm_compress_calls:>7} "
            f"{tr.compression_ms:>9.0f} "
            f"{tr.total_ms:>10.0f}"
        )
    print("=" * 78 + "\n")

    return traces



In [17]:
"""
    End-to-end Retrieve-Then-Compress RAG pipeline orchestrator.

    LLM CALL COUNT PER QUERY:
        Config 1: 1 call   (answer only)
        Config 2: 2 calls  (1 extractive compression + 1 answer)
        Config 3: 2 calls  (1 abstractive summarization + 1 answer)
        Config 4: 1 call   (answer only -- LLMLingua is local, zero API calls)
        Config 5: K + 1 + iterations*3 + 1 calls
                  (K MAP calls + 1 REDUCE + evid_iters*(1 check) + 1 answer)
                  Typical: 10 + 1 + 1 + 1 = 13 calls (K=10, 1 iteration)

    Config 5 is BEST for quality. Config 4 is BEST for cost+latency (zero compression API calls).
    Config 2 is BEST for source faithfulness (verbatim extraction, zero hallucination risk).
    """

'\n    End-to-end Retrieve-Then-Compress RAG pipeline orchestrator.\n\n    LLM CALL COUNT PER QUERY:\n        Config 1: 1 call   (answer only)\n        Config 2: 2 calls  (1 extractive compression + 1 answer)\n        Config 3: 2 calls  (1 abstractive summarization + 1 answer)\n        Config 4: 1 call   (answer only -- LLMLingua is local, zero API calls)\n        Config 5: K + 1 + iterations*3 + 1 calls\n                  (K MAP calls + 1 REDUCE + evid_iters*(1 check) + 1 answer)\n                  Typical: 10 + 1 + 1 + 1 = 13 calls (K=10, 1 iteration)\n\n    Config 5 is BEST for quality. Config 4 is BEST for cost+latency (zero compression API calls).\n    Config 2 is BEST for source faithfulness (verbatim extraction, zero hallucination risk).\n    '

In [18]:
config = RTCConfig()


In [19]:
pdf_paths    = download_pdfs(config)
chunks       = load_and_chunk_documents(pdf_paths, config)


2026-05-21 18:51:27 | INFO     | rtc_rag | Cached: attention_is_all_you_need.pdf  (2163.3 KB)
2026-05-21 18:51:27 | INFO     | rtc_rag | Cached: bert_pretraining_transformers.pdf  (757.0 KB)
2026-05-21 18:51:27 | INFO     | rtc_rag | Cached: rag_knowledge_intensive_nlp.pdf  (864.6 KB)
2026-05-21 18:51:29 | INFO     | rtc_rag |   attention_is_all_you_need.pdf -> 15 pages -> 104 chunks
2026-05-21 18:51:30 | INFO     | rtc_rag |   bert_pretraining_transformers.pdf -> 16 pages -> 173 chunks
2026-05-21 18:51:30 | INFO     | rtc_rag |   rag_knowledge_intensive_nlp.pdf -> 19 pages -> 181 chunks
2026-05-21 18:51:30 | INFO     | rtc_rag | Total chunks: 458


In [20]:
llm_compress = build_ollama_llm(config, temperature=config.LLM_COMPRESS_TEMPERATURE)


In [21]:
llm_answer   = build_ollama_llm(config, temperature=config.LLM_ANSWER_TEMPERATURE)


In [22]:
embeddings   = build_bge_embeddings(config)


2026-05-21 18:51:35 | INFO     | rtc_rag | Loading BGE: BAAI/bge-large-en-v1.5
2026-05-21 18:51:35 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5


C:\Users\dhanu\AppData\Local\Temp\ipykernel_21408\3458387679.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


2026-05-21 18:51:35 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 18:51:35 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-05-21 18:51:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


2026-05-21 18:51:36 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-21 18:51:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-21 18:51:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 18:51:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-21 18:51:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3741.63it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-21 18:51:38 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 18:51:38 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config.json "HTTP/1.1 200 OK"
2026-05-21 18:51:38 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 18:51:38 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-21 18:51:39 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-21 18:51:39 | INFO     | httpx |

In [23]:
vectorstore  = build_faiss_index(chunks, embeddings, config)


2026-05-21 18:51:40 | INFO     | rtc_rag | Loading FAISS from './faiss_rtc_index' ...
2026-05-21 18:51:40 | INFO     | faiss.loader | Loading faiss with AVX512 support.
2026-05-21 18:51:40 | INFO     | faiss.loader | Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-05-21 18:51:40 | INFO     | faiss.loader | Loading faiss with AVX2 support.
2026-05-21 18:51:40 | INFO     | faiss.loader | Successfully loaded faiss with AVX2 support.
2026-05-21 18:51:40 | INFO     | rtc_rag | FAISS loaded. Vectors: 458


In [31]:
bm25         = build_bm25_index(chunks, config)


2026-05-21 19:11:56 | INFO     | rtc_rag | Loading BM25 from 'bm25_rtc_index\bm25plus.pkl' ...
2026-05-21 19:11:56 | WARNING  | rtc_rag | BM25 cache invalid (Ran out of input). Rebuilding and overwriting cache.
2026-05-21 19:11:56 | INFO     | rtc_rag | Building BM25Plus from 458 chunks ...


In [32]:
demo_questions = [
        # Context-dense: answer is spread across multiple documents (abstractive excels)
        "What are the key innovations of the Transformer model compared to prior sequence models?",

        # Precise factual: single sentence contains the answer (extractive excels)
        "What BLEU score did the Transformer base model achieve on WMT 2014 English-to-German?",

        # Long-context multi-hop: requires synthesizing across sections (MapReduce excels)
        "How do the BERT and RAG models both use pre-trained transformers, and what are "
        "their fundamental differences in how they use retrieved information?",

        # Token budget critical: many partially-relevant docs need strict budget enforcement
        "What is the role of multi-head attention and how is it different from single-head?",

        # Evidentiality critical: answer exists but is scattered; loop may be needed
        "What training datasets and evaluation benchmarks are used in the Transformer paper?",
    ]


In [33]:
for question in demo_questions:
        run_all_configs(question, vectorstore, bm25, llm_compress, llm_answer, config)

logger.info("=== Retrieve-Then-Compress RAG Pipeline Demo Complete ===")



##############################################################################
QUERY: What are the key innovations of the Transformer model compared to prior sequence models?
##############################################################################

RUNNING: Config1_Baseline
2026-05-21 19:12:21 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

TRACE: Config1_Baseline_NoCompression
Query: What are the key innovations of the Transformer model compared to prior sequence

  Raw docs: 10  (4,326 chars)
  Compressed context: 1,888 chars
  Compression ratio: 2.29x
  LLM compress calls: 0
  Selective aug triggered: False

  Compressed context preview (first 300 chars):
  "[Attention Is All You Need p9]
Recurrent Neural Network Grammar [8].
In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the Berkeley-
Parser [29] even when training only on the WSJ training set of 40K sentences.
7 Conclusion
In this work, we presente